# Построение и обновление индексов

Notebook читает подготовленные Parquet, создаёт эмбеддинги, FAISS IVFPQ и BM25S-шарды, а также поддерживает инкрементальное обновление.

## Импорты и пути (необходимо установить свои пути!)

In [ ]:
%pip install -r requirements.txt

In [1]:
import gc
import os
import pickle
import re
import shutil
import time

import bm25s
import faiss
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

## Входные данные

В каталоге `data` ожидаются два Parquet-набора: `rag_documents_all.parquet` с полным корпусом и `rag_documents_new.parquet` с новыми документами текущего обновления. Оба набора содержат поля `id`, `req_desc`, `msg_pprb_chat` и `req_reg_date`. Все пути указаны относительно корня проекта.

In [2]:
BASE_DIR = "data"
# Полный накопительный Parquet. Используется только для первичного построения RAG.
RAG_ALL_PARQUET_PATH = (f"{BASE_DIR}/rag_documents_all.parquet")
# Parquet только с новыми ID текущего запуска. Используется для обновления существующего RAG.
RAG_NEW_PARQUET_PATH = (f"{BASE_DIR}/rag_documents_new.parquet")
# Путь, где будем лежать необходимый кэш
path_to_save = ("cache_le_finale2")
path_to_load = path_to_save


ID_COL = "id"
REQ_DESC_COL = "req_desc"
CHAT_COL = "msg_pprb_chat"
DATE_COL = "req_reg_date"
BASE_COLS = [ID_COL, REQ_DESC_COL, CHAT_COL, DATE_COL]
BM25_CHUNK = 1_000_000
EMBED_CHUNK_SIZE = 10_000
EMBED_BATCH_SIZE = 48
FAISS_M = 32
FAISS_NBITS = 8

documents = []
doc_ids = []
tokenized_corpus = []
id_to_index = {}
req_descs = []
msg_pprb_chats = []
req_reg_dates = []

In [3]:
PATH_TO_EMBED_MODEL = os.getenv("EMBEDDING_MODEL_NAME", "BAAI/bge-m3")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print (DEVICE)
print("Загрузка BGE-NЗ...")
EMBED_MODEL = SentenceTransformer(PATH_TO_EMBED_MODEL, device=DEVICE)
EMBED_MODEL.to(DEVICE)
DIM = EMBED_MODEL.get_sentence_embedding_dimension()
print(f"Готово. Размерность вектора: (DIM)")

cpu
Загрузка BGE-NЗ...


Готово. Размерность вектора: (DIM)


## 1. Основная часть: первичное создание эмбеддингов, токенизация текстов, первичное построение faiss индекса и bm-25 шардов, сохранение и загрузка меты

Вспомогательные функции

In [4]:
def build_text(req_desc: str, msg_pprb_chat: str):
    """Объединяет описание и текст диалога."""
    parts = []
    if req_desc and str(req_desc).strip():
        parts.append(str(req_desc))
    if msg_pprb_chat and str(msg_pprb_chat).strip():
        parts.append(str(msg_pprb_chat))
    return " ".join(parts)


def tokenize(text):
    """Разбивает текст на токены."""
    return re.findall(r"[а-яёa-z0-9]+", text.lower())


def embed(texts, batch_size=48, log_every=50000):
    """Создаёт эмбеддинги для списка текстов."""
    all_embeddings = []
    total = len(texts)
    start_total = time.time()
    last_log_time = start_total
    processed = 0
    last_logged_processed = 0
    next_log = log_every
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        emb = EMBED_MODEL.encode(
            batch,
            normalize_embeddings=True,
            convert_to_numpy=True,
            batch_size=batch_size,
            device=DEVICE,
            chunk_size=200,
            show_progress_bar=False
        )
        all_embeddings.append(emb)
        processed += len(batch)
        if processed >= next_log or processed == total:
            now = time.time()
            last_log_time = now
            last_logged_processed = processed
            while next_log <= processed:
                next_log += log_every
        # Оценка каждые ~100к эмбеддингов
        if len(all_embeddings) >= 10:
            all_embeddings = [np.vstack(all_embeddings)]
    return np.vstack(all_embeddings).astype("float32")

Построение эмбеддингов и привязка эмбеддингов к объектам нового датасета (создание массивов с метаданными)
Используется только при первоначальном построении (если ничего обученного еще нет)

In [5]:
def prepare_documents(data):
    """Формирует документы и полную мету корпуса."""
    global documents, doc_ids, tokenized_corpus, id_to_index, req_descs, msg_pprb_chats, req_reg_dates
    start_total = time.time()
    documents = []
    doc_ids = []
    tokenized_corpus = []
    id_to_index = {}
    req_descs = []
    msg_pprb_chats = []
    req_reg_dates = []

    def process_row(row):
        """Добавляет одну строку в полную мету."""
        text = build_text(row[REQ_DESC_COL], row[CHAT_COL])
        documents.append(text)
        cid = row[ID_COL]
        doc_ids.append(cid)
        id_to_index[cid] = len(documents) - 1
        tokenized_corpus.append(tokenize(text))
        req_descs.append(row[REQ_DESC_COL])
        msg_pprb_chats.append(row[CHAT_COL])
        dt = pd.to_datetime(row.get(DATE_COL, None), errors="coerce")
        if pd.isna(dt):
            req_reg_dates.append(None)
        else:
            req_reg_dates.append(dt.strftime("%Y-%m-%d"))
        return text

    data.apply(process_row, axis=1)
    print(f"Обработано {len(documents)} документов.")
    print(f"[TIME] Подготовка текстов: {time.time() - start_total:.2f} сек")
    return documents


def build_and_save_embeddings(texts, path=path_to_save):
    """Создаёт и сохраняет эмбеддинги в memmap."""
    os.makedirs(path, exist_ok=True)
    t0 = time.time()
    print("Начинаем построение эмбеддингов...")
    dim = EMBED_MODEL.get_sentence_embedding_dimension()
    total_docs = len(texts)
    embeddings_path = f"{path}/embeddings.memmap"
    embeddings = np.memmap(embeddings_path, dtype="float32", mode="w+", shape=(total_docs, dim))
    print(f"Создан memmap: {total_docs} x {dim} (float32)")
    chunk_size = 10000  
    start_idx = 0

    for i in range(0, total_docs, chunk_size):
        chunk = texts[i:i + chunk_size]
        print(f"Обработка чанка {i}-{i+len(chunk)}...")
        chunk_emb = embed(chunk, batch_size=48, log_every=5000)
        # Записываем в memmap
        end_idx = start_idx + len(chunk_emb)
        embeddings[start_idx:end_idx] = chunk_emb
        embeddings.flush()  # сбрасываем на диск
        print(f"[MEMMAP] Записано {end_idx}/{total_docs} эмбеддингов")
        # Очистка памяти
        del chunk_emb
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        start_idx = end_idx

    # Сохраняем мета-информацию
    with open(f"{path}/embeddings_meta.pkl", "wb") as f:
        pickle.dump({"shape": (total_docs, dim), "dtype": "float32", "path": embeddings_path}, f)
    print("Эмбеддинги успешно записаны на диск (memmap)")
    print(f"[TIME] Построение эмбеддингов: {time.time() - t0:.2f} сек")
    return embeddings


def save_meta(path=path_to_save):
    """Сохраняет полную мету в файл."""
    os.makedirs(path, exist_ok=True)
    with open(f"{path}/meta.pkl", "wb") as f:
        pickle.dump({
            "documents": documents,
            "doc_ids": doc_ids,
            "tokenized_corpus": tokenized_corpus,
            "id_to_index": id_to_index,
            "req_descs": req_descs,
            "msg_pprb_chats": msg_pprb_chats,
            "req_reg_dates": req_reg_dates,
        }, f)
    print("мета сохранена")

In [6]:
def load_embeddings(path=path_to_load):
    """Загружает эмбеддинги через memmap."""
    meta_path = f"{path}/embeddings_meta.pkl"
    if os.path.exists(meta_path):
        with open(meta_path, "rb") as f:
            meta = pickle.load(f)
        embeddings = np.memmap(meta["path"], dtype=meta["dtype"], mode="r", shape=meta["shape"])
        print(f"Эмбеддинги загружены через memmap: {meta['shape']}")
        return embeddings

def load_meta(path=path_to_load):
    """Загружает полную мету из файла."""
    global documents, doc_ids, tokenized_corpus, id_to_index, req_descs, msg_pprb_chats, req_reg_dates
    with open(f"{path}/meta.pkl", "rb") as f:
        meta = pickle.load(f)
    documents = meta["documents"]
    doc_ids = meta["doc_ids"]
    tokenized_corpus = meta["tokenized_corpus"]
    id_to_index = meta["id_to_index"]
    req_descs = meta["req_descs"]
    msg_pprb_chats = meta["msg_pprb_chats"]
    req_reg_dates = meta.get("req_reg_dates", [None] * len(documents))
    print("мета успешно загружена!")
    print(f"Количество документов: {len(documents)}")

Построение FAISS индекса (используется только при первоначальном построении, когда FAISS индекса еще вообще нет или при faiss_mode="rebuild")

In [7]:
def build_and_save_faiss_index(embeddings, FAISS_M=FAISS_M, FAISS_NBITS=FAISS_NBITS, TRAIN_SAMPLE_SIZE=2500000, FAISS_ADD_BATCH_SIZE=10000, FAISS_NLIST=None, FAISS_NPROBE=None, path=path_to_save):
    """Строит и сохраняет индекс FAISS IVFPQ."""
    t0 = time.time()
    dim = embeddings.shape[1]
    n = len(documents)
    if n <= 50_000_000:
        default_nlist = min(12000, max(128, int(2 * np.sqrt(n)))) if n > 0 else 1
        nlist = FAISS_NLIST if FAISS_NLIST is not None else default_nlist
        # для тестового прогона:
        # nlist = min(128, max(32, int(np.sqrt(n)))) if n > 0 else 1
        sample_size = min(TRAIN_SAMPLE_SIZE, len(embeddings))
        # для тестового прогона
        # sample_size = min(3000, len(embeddings))
    else:
        default_nlist = max(12000, max(128, int(2 * np.sqrt(n))))
        nlist = FAISS_NLIST if FAISS_NLIST is not None else default_nlist
        sample_size = min(
            len(embeddings),
            max(TRAIN_SAMPLE_SIZE, nlist * 100),
            4_000_000
        )

    quantizer = faiss.IndexFlatL2(dim)
    index = faiss.IndexIVFPQ(quantizer, dim, nlist, FAISS_M, FAISS_NBITS)
    print(f"[TIME] Создание FAISS: {time.time() - t0:.2f} сек")
    if DEVICE == "cuda":
        try:
            res = faiss.StandardGpuResources()
            # res.setTempMemory(1024 * 1024 * 1024)
            index = faiss.index_cpu_to_gpu(res, 0, index)
            print("FAISS перенесён на GPU.")
            print("Устройство:", type(index))
        except Exception as e:
            print("Не удалось перенести FAISS на GPU, остаёмся на CPU")
            print("Ошибка:", e)
    else:
        print("FAISS работает на CPU.")
    print(f"[TIME] Перенос FAISS на GPU: {time.time() - t0:.2f} сек")

    # Train sample
    train_ids = np.random.choice(len(embeddings), sample_size, replace=False)
    train_data = np.ascontiguousarray(embeddings[train_ids].astype("float32"))
    print("sample_size для обучения:", sample_size)
    print("train_data shape:", train_data.shape)
    print(f"[TIME] Подготовка train_data: {time.time() - t0:.2f} сек")

    # FAISS train
    t0 = time.time()
    print("Начинаем обучение FAISS index.train(...)")
    index.train(train_data)
    print(f"FAISS train завершён. [TIME] Обучение FAISS / кластеризация: {time.time() - t0:.2f} сек")
    del train_data
    del train_ids
    gc.collect()

    # FAISS add
    t0 = time.time()
    print("Начинаем добавление векторов в индекс...")
    for i in range(0, len(embeddings), FAISS_ADD_BATCH_SIZE):
        batch = np.ascontiguousarray(embeddings[i:i + FAISS_ADD_BATCH_SIZE].astype("float32"))
        index.add(batch)
        # Если вы хотите видеть прогресс хотя бы раз в 10-20 секунд, можно добавить простую печать:
        if i % 1000000 == 0:  # Печатаем каждые 100k документов
            print(f"Добавлено {i} из {len(embeddings)} векторов...")
    print("Добавление векторов завершено.")
    print("index.ntotal:", index.ntotal)
    print(f"[TIME] Добавление векторов FAISS index.add: {time.time() - t0:.2f} сек")
    # nprobe
    default_nprobe = min(256, max(32, int(nlist * 0.02)))
    index.nprobe = FAISS_NPROBE if FAISS_NPROBE is not None else default_nprobe
    # для тестового прогона
    # index.nprobe = min(16, nlist)
    print("index.nprobe:", index.nprobe)
    try:
        faiss_index_to_save = faiss.index_gpu_to_cpu(index)
        print("FAISS index перенесён с GPU на CPU для сохранения.")
    except Exception:
        faiss_index_to_save = index
        print("FAISS index уже на CPU или перенос не требуется")
        
    print(f"Тип индекса после переноса: {type(faiss_index_to_save)}")
    # Сохраняем индекс
    faiss.write_index(faiss_index_to_save, f"{path}/faiss_index")
    print("FAISS построен и сохранён")
    return index

Построение BM25-шардов (используется только при первоначальном построении, когда никаких шардов еще вообще нет)

In [8]:
def build_bm25s_shards(tokenized_corpus, chunk_size=BM25_CHUNK, out_dir=path_to_save):
    """Строит и сохраняет BM25S-шарды."""
    os.makedirs(out_dir, exist_ok=True)
    for i in range(0, len(tokenized_corpus), chunk_size):
        shard_id = i // chunk_size
        retriever = bm25s.BM25()
        retriever.index(tokenized_corpus[i:i+chunk_size], show_progress=False)
        retriever.save(f"{out_dir}/bm25s_shards2/shard_{shard_id}")
        print(f"✅ BM25 shard_{shard_id} сохранён ({len(tokenized_corpus[i:i+chunk_size])} док)")
        del retriever; gc.collect()

## 2. Основная часть

Чтение Parquet из каталога проекта

In [9]:
def read_all_data_from_parquet(parquet_path=RAG_ALL_PARQUET_PATH):
    """Читает полный корпус из Parquet."""
    all_data = pd.read_parquet(parquet_path, columns=BASE_COLS)
    print("Строк в полном накопительном Parquet:", len(all_data))
    return all_data


def read_current_new_data_from_parquet(parquet_path=RAG_NEW_PARQUET_PATH):
    """Читает новые документы из Parquet."""
    if (parquet_path is None or not os.path.exists(parquet_path)):
        print("Parquet с новыми данными не найден")
        return pd.DataFrame(columns=BASE_COLS)
    new_data = pd.read_parquet(parquet_path, columns=BASE_COLS)
    new_data[ID_COL] = (new_data[ID_COL].astype(str).str.strip())
    print("Строк в Parquet с новыми данными:", len(new_data))
    return new_data

Добавление новых документов в мета-переменные

In [10]:
# В отличие от prepare_documents(), эта функция не очищает существующие списки, а дописывает в них только новые ID
def prepare_new_documents(data):
    """Добавляет в мету документы с новыми ID."""
    global documents, doc_ids, tokenized_corpus, id_to_index, req_descs, msg_pprb_chats, req_reg_dates
    start_total = time.time()
    existing_ids = set(doc_ids)
    new_texts = []

    def process_row(row):
        """Добавляет одну новую строку в мету."""
        if pd.isna(row[ID_COL]):
            return None
        cid = str(row[ID_COL]).strip()
        # Повторная защита от дублей. Даже если в новом Parquet случайно находится старый ID, повторно он не индексируется.
        if (not cid or cid in existing_ids):
            return None
        text = build_text(row[REQ_DESC_COL], row[CHAT_COL])
        documents.append(text)
        doc_ids.append(cid)
        tokenized_corpus.append(tokenize(text))
        req_descs.append(row[REQ_DESC_COL])
        msg_pprb_chats.append(row[CHAT_COL])
        dt = pd.to_datetime(row.get(DATE_COL, None), errors="coerce")
        if pd.isna(dt):
            req_reg_dates.append(None)
        else:
            req_reg_dates.append(dt.strftime("%Y-%m-%d"))
        id_to_index[cid] = len(documents) - 1
        existing_ids.add(cid)
        new_texts.append(text)
        return text

    data.apply(process_row, axis=1)
    print(f"Обработано {len(new_texts)} новых документов для индексации")
    print(f"[TIME] Подготовка текстов: {time.time() - start_total:.2f} сек")
    return new_texts

Дописывает эмбеддинги новых документов в общий memmap; добавляет их в существующий индекс FAISS, либо полностью его перестраивает (в зависимости от режима add/rebuild)

In [11]:
def append_embeddings_and_update_faiss(new_texts, old_n, faiss_mode="add", path=path_to_save):
    """Дописывает эмбеддинги и обновляет FAISS."""
    if faiss_mode not in ("add", "rebuild"):
        raise ValueError("faiss_mode должен быть 'add' или 'rebuild'")
    if not new_texts:
        return faiss.read_index(f"{path}/faiss_index")

    embeddings_meta_path = (f"{path}/embeddings_meta.pkl")
    with open(embeddings_meta_path, "rb") as file:
        embeddings_meta = (pickle.load(file))
    old_shape = tuple(embeddings_meta["shape"])
    old_embeddings_path = (embeddings_meta["path"])
    old_embeddings_count, dim = (old_shape)
    if old_embeddings_count != old_n:
        raise RuntimeError(f"Количество старых embeddings не совпадает с meta: embeddings={old_embeddings_count}, meta={old_n}")
    expected_total = (old_n + len(new_texts))
    # Существующий memmap нельзя просто увеличить, поэтому сначала создаётся временный файл.
    temporary_path = (old_embeddings_path + ".tmp")
    if os.path.exists(temporary_path):
        os.remove(temporary_path)
    old_embeddings = np.memmap(old_embeddings_path, dtype=embeddings_meta["dtype"], mode="r", shape=old_shape)
    enlarged_embeddings = np.memmap(temporary_path, dtype="float32", mode="w+", shape=(expected_total, dim))
    print("Копирование старых embeddings в новый memmap...")
    for start in range(0, old_n, 100_000):
        end = min(start + 100_000, old_n)
        enlarged_embeddings[start:end] = old_embeddings[start:end]

    # При режиме add загружаем существующий FAISS.
    if faiss_mode == "add":
        index = faiss.read_index(f"{path}/faiss_index")
        if index.ntotal != old_n:
            raise RuntimeError(f"Количество старых векторов FAISS не совпадает с meta: faiss={index.ntotal}, meta={old_n}")
        try:
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
            print("FAISS перенесён на GPU")
        except Exception as error:
            print("Не удалось перенести FAISS на GPU, остаёмся на CPU")
            print("Ошибка:", error)

    # Создаем эмбеддинги только для новых текстов
    for start in range(0, len(new_texts), EMBED_CHUNK_SIZE):  # разделяем новые текста на внешние чанки
        end = min(start + EMBED_CHUNK_SIZE, len(new_texts))  # конец текущего чанка
        text_chunk = new_texts[start:end]
        print(f"Создание новых эмбеддингов: {start}:{end}")
        chunk_embeddings = embed(text_chunk, batch_size=EMBED_BATCH_SIZE, log_every=5000)
        # расчет позиции новых эмбеддингов
        target_start = (old_n + start)
        target_end = (old_n + end)
        # запись в новый memmap
        enlarged_embeddings[target_start:target_end] = chunk_embeddings
        # сброс данных на диск
        enlarged_embeddings.flush()
        # В режиме add новые векторы сразу добавляются в существующий индекс.
        if faiss_mode == "add":
            index.add(np.ascontiguousarray(chunk_embeddings.astype("float32")))
        del chunk_embeddings
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if faiss_mode == "rebuild":
        # Полное переобучение FAISS на старых и новых эмбеддингах.
        index = (build_and_save_faiss_index(enlarged_embeddings, path=path))
    else:
        try:
            index_to_save = (faiss.index_gpu_to_cpu(index))
        except Exception:
            index_to_save = index
        faiss.write_index(index_to_save, f"{path}/faiss_index")
        print("FAISS построен и сохранён")

    enlarged_embeddings.flush()
    del old_embeddings
    del enlarged_embeddings
    gc.collect()
    # Только после успешного создания нового файла заменяем старый memmap.
    os.replace(temporary_path, old_embeddings_path)
    with open(embeddings_meta_path, "wb") as file:
        pickle.dump({"shape": (expected_total, dim), "dtype": "float32", "path": (old_embeddings_path)}, file)

    print("Добавлено новых embeddings:", len(new_texts))
    print("FAISS index.ntotal:", index.ntotal)
    return index

Обновление BM-25 шардов (дозапись)

In [12]:
# Если последний старый шард был заполнен не полностью, он перестраивается с начала этого шарда. Полностью готовые предыдущие шарды не трогаются.
def append_bm25s_shards(old_n, chunk_size=BM25_CHUNK, out_dir=path_to_save):
    """Перестраивает затронутые BM25S-шарды."""
    new_total = len(tokenized_corpus)  # tokenized_corpus к этому моменту содержит токены всех обращений (включая новые и старые)
    # проверяем, появились ли новые документы
    if new_total <= old_n:
        return
    # определяем, с какой позиции перестраивать; например, для old_n = 7 903 032 и chunk_size = 1000 000 имеем (old_n // chunk_size) * chunk_size = 7 000 000; значит перестраиваем с 7 шарда,
    # поскольку он был заполнен не полностью и новые документы должны попасть в него
    if old_n % chunk_size == 0:  # особый случай: последний шард был заполнен полностью (old_n = 8 000 000, например)
        start_position = old_n
    else:
        start_position = (old_n // chunk_size) * chunk_size
    shards_dir = (f"{out_dir}/bm25s_shards2")
    os.makedirs(shards_dir, exist_ok=True)
    for i in range(start_position, new_total, chunk_size):
        # определяем номер шарда и его границы
        shard_id = (i // chunk_size)
        end = min(i + chunk_size, new_total)
        shard_path = (f"{shards_dir}/shard_{shard_id}")
        # Пересоздаваем только затронутый шард (удаляем старую версию затронутого шарда, тк BM25S не дописывает документы внутрь уже сохраненного индекса, потому последний неполный индекс
        # необходимо удалить и создать заново
        if os.path.isdir(shard_path):
            shutil.rmtree(shard_path)
        elif os.path.exists(shard_path):
            os.remove(shard_path)
        # строим обновленный BM25
        retriever = bm25s.BM25()
        retriever.index(tokenized_corpus[i:end], show_progress=False)
        retriever.save(shard_path)
        print(f"BM25 shard_{shard_id} обновлён ({end - i} документов)")
        del retriever
        gc.collect()

Проверка согласованности кэша

In [13]:
def verify_saved_state(path=path_to_load):
    """Проверяет согласованность сохранённых артефактов."""
    with open(f"{path}/embeddings_meta.pkl", "rb") as file:
        embeddings_meta = (pickle.load(file))

    embeddings_count = int(embeddings_meta["shape"][0])
    faiss_index = faiss.read_index(f"{path}/faiss_index")
    faiss_count = int(faiss_index.ntotal)
    documents_count = len(documents)
    ids_count = len(doc_ids)
    tokens_count = len(tokenized_corpus)
    if not (embeddings_count == faiss_count == documents_count == ids_count == tokens_count):
        raise RuntimeError(f"Нарушено соответствие cache: embeddings={embeddings_count}, faiss={faiss_count}, documents={documents_count}, doc_ids={ids_count}, tokenized_corpus={tokens_count}")
    print("Проверка cache пройдена:", documents_count)

## Финальный запуск (выберите сценарий который нужен вам!)

Обновление существующей базы новыми обращениями

In [14]:
def update_existing_rag(new_data, faiss_mode="add", path=path_to_save):
    """Обновляет мету, эмбеддинги и индексы."""
    load_meta(path)
    # Проверяем состояние базы до обновления.
    verify_saved_state(path)
    old_n = len(documents)
    # Добавляются только ID, которых ещё нет в мете.
    new_texts = prepare_new_documents(new_data)
    if not new_texts:
        print("Новых ID нет, индексы не изменены")
        return faiss.read_index(f"{path}/faiss_index")
    index = (append_embeddings_and_update_faiss(new_texts=new_texts, old_n=old_n, faiss_mode=faiss_mode, path=path))
    append_bm25s_shards(old_n=old_n, out_dir=path)
    save_meta(path)
    # Проверяем состояние после обновления.
    verify_saved_state(path)
    print("ОБНОВЛЕНИЕ RAG ЗАВЕРШЕНО")
    print("Документов было:", old_n)
    print("Добавлено новых документов:", len(new_texts))
    print("Документов стало:", len(documents))
    print("=" * 60)
    return index

Первичное построение для демонстрационного корпуса (используйте, когда готового cache ещё нет)

In [15]:
def initial_build(data, FAISS_NLIST=None, FAISS_NPROBE=None, FAISS_M=FAISS_M, FAISS_NBITS=FAISS_NBITS, TRAIN_SAMPLE_SIZE=2500000, path=path_to_save):
    """Выполняет первоначальное построение индексов."""
    texts = prepare_documents(data)
    embeddings = (build_and_save_embeddings(texts, path=path))
    index = (build_and_save_faiss_index(embeddings, FAISS_M=FAISS_M, FAISS_NBITS=FAISS_NBITS, TRAIN_SAMPLE_SIZE=TRAIN_SAMPLE_SIZE, FAISS_NLIST=FAISS_NLIST, FAISS_NPROBE=FAISS_NPROBE, path=path))
    build_bm25s_shards(tokenized_corpus, out_dir=path)
    save_meta(path)
    verify_saved_state(path)
    print("ПЕРВИЧНОЕ ПОСТРОЕНИЕ RAG ЗАВЕРШЕНО")
    print("Документов:", len(documents))
    return index

Запуск (выберите один из трёх режимов через параметр `mode`)

In [16]:
def run_indexing(mode, FAISS_NLIST=None, FAISS_NPROBE=None, FAISS_NBITS=FAISS_NBITS, TRAIN_SAMPLE_SIZE=2500000):
    """Запускает выбранный режим построения или обновления индексов."""
    # ВАРИАНТ А. ПЕРВИЧНОЕ ПОСТРОЕНИЕ
    # Запускаемся только тогда, когда готового cache ещё нет. Для демонстрации ниже используется этот режим.
    if mode == "initial":
        all_data = read_all_data_from_parquet(RAG_ALL_PARQUET_PATH)
        return initial_build(all_data, FAISS_NLIST=FAISS_NLIST, FAISS_NPROBE=FAISS_NPROBE, FAISS_NBITS=FAISS_NBITS, TRAIN_SAMPLE_SIZE=TRAIN_SAMPLE_SIZE)

    # ВАРИАНТ B. ОБЫЧНОЕ ПОПОЛНЕНИЕ
    # Эмбеддинги создаются только для новых ID. Существующий FAISS не обучается заново, используется index.add().
    # Что значит обучать FAISS заново? Это означает создавать заново кластеры внутри кластеризованного индекса. Если переводить на простой язык, то, если пополнение данных не огромно (в текущей
    # базе около 8 миллионов обращений), то есть не кратно больше исходной базы) ваши данные на примерно одну и ту же тематику, то запустить этот вариант
    if mode == "add":
        new_data = read_current_new_data_from_parquet(RAG_NEW_PARQUET_PATH)
        return update_existing_rag(new_data, faiss_mode="add")

    # ВАРИАНТ C. ОБНОВЛЕНИЕ С ПОЛНЫМ ПЕРЕОБУЧЕНИЕМ FAISS
    # Использовать только осознанно: на миллионах документов это значительно дольше и требует больше памяти.
    # РЕКОМЕНДУЕТСЯ ИСПОЛЬЗОВАТЬ ЕГО, ЕСЛИ:
    # -появилось сильно больше 8 млн обращений (условно еще 8 миллионов)
    # -данные обновились тематически (появились обращения по совершенно новой тематике)
    if mode == "rebuild":
        new_data = read_current_new_data_from_parquet(RAG_NEW_PARQUET_PATH)
        return update_existing_rag(new_data, faiss_mode="rebuild")

    raise ValueError("mode должен быть 'initial', 'add' или 'rebuild'")

In [17]:
# Для демо на 100 строк передаём уменьшенные параметры FAISS.
# Для масштабного корпуса их можно не указывать: будут использованы исходные значения.

In [18]:
faiss_index = run_indexing(
    mode="initial",
    FAISS_NLIST=2,
    FAISS_NPROBE=2,
    FAISS_NBITS=1,
    TRAIN_SAMPLE_SIZE=100,
)

Строк в полном накопительном Parquet: 100
Обработано 100 документов.
[TIME] Подготовка текстов: 0.00 сек
Начинаем построение эмбеддингов...
Создан memmap: 100 x 1024 (float32)
Обработка чанка 0-100...


[MEMMAP] Записано 100/100 эмбеддингов
Эмбеддинги успешно записаны на диск (memmap)
[TIME] Построение эмбеддингов: 4.68 сек
[TIME] Создание FAISS: 0.00 сек
FAISS работает на CPU.
[TIME] Перенос FAISS на GPU: 0.00 сек
sample_size для обучения: 100
train_data shape: (100, 1024)
[TIME] Подготовка train_data: 0.00 сек
Начинаем обучение FAISS index.train(...)
FAISS train завершён. [TIME] Обучение FAISS / кластеризация: 0.01 сек
Начинаем добавление векторов в индекс...
Добавлено 0 из 100 векторов...
Добавление векторов завершено.
index.ntotal: 100
[TIME] Добавление векторов FAISS index.add: 0.00 сек
index.nprobe: 2
FAISS index уже на CPU или перенос не требуется
Тип индекса после переноса: <class 'faiss.swigfaiss.IndexIVFPQ'>
FAISS построен и сохранён
✅ BM25 shard_0 сохранён (100 док)
мета сохранена
Проверка cache пройдена: 100
ПЕРВИЧНОЕ ПОСТРОЕНИЕ RAG ЗАВЕРШЕНО
Документов: 100
